# Mart: IP charges (annual time-grain) fact table

Reads the staged OECD IP charges data (long — one row per accounting entry) and
writes the final dashboard-ready **wide** fact table (one row per year × counterpart).

| | |
|---|---|
| **Reads** | `data/staging/oecd/stg_oecd__ip_charges.csv` |
| **Writes** | `data/marts/fct_ip_charges_yearly.csv` |

**Output schema** (columns follow the OECD Data Explorer's general-audience labels):

| Column | Type | Description |
|---|---|---|
| `year` | int | Reference year |
| `counterpart_country` | str | Trading partner full name (e.g. `"United States"`) |
| `balance_cad_millions` | float | Net = revenue − expenditure, millions CAD (negative = Canada paid more) |
| `revenue_cad_millions` | float | Revenue: royalties Canada **earns**, millions CAD (null for some partner-years) |
| `expenditure_cad_millions` | float | Expenditure: royalties Canada **pays**, millions CAD (null for some partner-years) |

In [1]:
import pandas as pd
from pathlib import Path
import sys

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

from utils.paths import STG_OECD_DIR, MARTS_DIR
from utils.oecd_utils import pivot_ip_charges_wide

MARTS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
INPUT_FILENAME  = 'stg_oecd__ip_charges.csv'
OUTPUT_FILENAME = 'fct_ip_charges_yearly.csv'

df = pd.read_csv(STG_OECD_DIR / INPUT_FILENAME)
print(f'Loaded {len(df)} rows from staging (long: one row per accounting entry)')
df.head()

Loaded 3953 rows from staging (long: one row per accounting entry)


,year,counterpart_area,counterpart_country,accounting_entry,value_cad_millions
0,1969,W,World,B,-167.0
1,1969,W,World,C,4.0
2,1969,W,World,D,171.0
3,1970,W,World,B,0.0
4,1970,W,World,C,0.0


In [3]:
# 1) Pivot the long staged data (one row per accounting entry) into the wide fact
#    shape: one row per (year, counterpart) with balance/revenue/expenditure columns.
wide = pivot_ip_charges_wide(df)

# 2) Align OECD country names to the Dim MacroRegion naming convention in Power BI.
#    Mapping uses ISO area codes (counterpart_area) to avoid special-character issues.
AREA_CODE_NAME_MAP = {
    'CHN': 'China',
    'HKG': 'Hong Kong',
    'SVK': 'Slovakia',
    'KOR': 'South Korea',
    'TWN': 'Taiwan',
}

fct_df = (
    wide
    .assign(counterpart_country=lambda x: x['counterpart_area'].map(AREA_CODE_NAME_MAP).fillna(x['counterpart_country']))
    [['year', 'counterpart_country',
      'balance_cad_millions', 'revenue_cad_millions', 'expenditure_cad_millions']]
    .sort_values(['year', 'counterpart_country'])
    .reset_index(drop=True)
)

print(f'Final mart shape: {fct_df.shape}')
print(f'Countries: {sorted(fct_df["counterpart_country"].unique().tolist())}')
fct_df.head(10)

Final mart shape: (1323, 5)
Countries: ['Argentina', 'Australia', 'Austria', 'Bahamas', 'Barbados', 'Belgium', 'Bermuda', 'Brazil', 'British Virgin Islands', 'Brunei Darussalam', 'Bulgaria', 'Cayman Islands', 'Chile', 'China', 'Colombia', 'Costa Rica', 'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'Ecuador', 'Egypt', 'El Salvador', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 'Guatemala', 'Hong Kong', 'Hungary', 'Iceland', 'India', 'Indonesia', 'Ireland', 'Israel', 'Italy', 'Jamaica', 'Japan', 'Kazakhstan', 'Kenya', 'Latvia', 'Lithuania', 'Luxembourg', 'Malaysia', 'Malta', 'Mauritius', 'Mexico', 'Netherlands', 'New Zealand', 'Nigeria', 'Norway', 'Oman', 'Pakistan', 'Panama', 'Papua New Guinea', 'Peru', 'Philippines', 'Poland', 'Portugal', 'Qatar', 'Romania', 'Russia', 'Saudi Arabia', 'Singapore', 'Slovakia', 'Slovenia', 'South Africa', 'South Korea', 'Spain', 'Sri Lanka', 'Sweden', 'Switzerland', 'Taiwan', 'Thailand', 'Trinidad and Tobago', 'Türkiye', 'Ukraine', 'United Arab Emira

,year,counterpart_country,balance_cad_millions,revenue_cad_millions,expenditure_cad_millions
0,1969,World,-167.0,4.0,171.0
1,1970,World,0.0,0.0,0.0
2,1971,World,0.0,0.0,0.0
3,1972,World,0.0,0.0,0.0
4,1973,World,-243.0,7.0,250.0
5,1974,World,0.0,0.0,0.0
6,1975,World,0.0,0.0,0.0
7,1976,World,0.0,0.0,0.0
8,1977,World,-410.0,32.0,442.0
9,1978,World,0.0,0.0,0.0


In [4]:
output_path = MARTS_DIR / OUTPUT_FILENAME
fct_df.to_csv(output_path, index=False, encoding='utf-8')
print(f'Fact table saved to: {output_path}')

Fact table saved to: C:\Users\GuidaK\OneDrive - ISED-ISDE\Documents\Work\or_country_profiles_dashboard\data\marts\fct_ip_charges_yearly.csv
